<a href="https://colab.research.google.com/github/Jake0925/LLM/blob/main/LLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1️⃣ 라이브러리 설치

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00


보안토큰 사용

In [2]:
from huggingface_hub import login
from google.colab import userdata

# Colab 보안 비밀에서 HF_TOKEN을 가져옵니다.
hf_token = userdata.get('HF_TOKEN')

# Hugging Face에 로그인합니다.
login(token=hf_token)

print("Hugging Face 로그인에 성공했습니다!")

Hugging Face 로그인에 성공했습니다!


접근권한상태 확인


In [3]:
from huggingface_hub import model_info


try:
    # 모델 정보를 가져와서 접근 권한이 있는지 확인합니다.
    info = model_info("meta-llama/Llama-2-7b-hf", token=userdata.get('HF_TOKEN'))
    print(f"✅ 모델 접근 권한이 확인되었습니다! 모델 ID: {info.id}")
except Exception as e:
    if "403" in str(e):
        print("❌ 아직 승인 대기 중이거나 거절되었습니다. Hugging Face 페이지에서 승인 여부를 확인하세요.")
    elif "401" in str(e):
        print("❌ 토큰이 유효하지 않습니다. Secrets 설정을 다시 확인해 주세요.")
    else:
        print(f"❌ 오류 발생: {e}")

✅ 모델 접근 권한이 확인되었습니다! 모델 ID: meta-llama/Llama-2-7b-hf


# 2️⃣ 모델 불러오기 (QLoRA 방식)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from google.colab import userdata

# 모델 이름 설정
model_name = "meta-llama/Llama-2-7b-hf"
hf_token = userdata.get('HF_TOKEN')

# 1. 4bit 양자화 설정 (BitsAndBytesConfig 사용)
# -  70억 개의 파라미터를 가진 거대 모델을 그대로 불러오면 메모리가 부족합니다. 이를 4비트로 압축(양자화)하여 메모리 사용량을 획기적으로 줄이는 설정입니다.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. 토크나이저 불러오기
# - 텍스트를 모델이 이해할 수 있는 숫자로 변환해주는 도구
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token
)

# 3. 모델 불러오기 (quantization_config 전달)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # GPU 메모리에 모델을 자동으로 적절히 배치
    token=hf_token,
    torch_dtype=torch.float16
)

print(f"{model_name} 모델과 토크나이저를 성공적으로 불러왔습니다!")

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

meta-llama/Llama-2-7b-hf 모델과 토크나이저를 성공적으로 불러왔습니다!



* 오류 \
TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'에 대해 설명해 드리겠습니다.


* 오류 원인 \
transformers 라이브러리가 업데이트되면서, from_pretrained 함수에 load_in_4bit=True와 같은 양자화 설정을 직접 전달하는 방식이 더 이상 모델 초기화 함수(__init__)에서 지원되지 않게 되었습니다.

해결 방법 \
이제는 **BitsAndBytesConfig**라는 별도의 설정 객체를 만들어 양자화 옵션들을 담은 뒤, 이를 **quantization_config**라는 인자로 모델에 전달해야 합니다.

현재 Bk9YiISIqXsM 셀에 수정해 드린 코드는 이 최신 방식을 적용하여 다음과 같이 변경되었습니다:

BitsAndBytesConfig를 생성하여 4bit 설정값을 저장합니다.
from_pretrained 호출 시 quantization_config=bnb_config를 사용합니다.
이제 셀을 실행하시면 정상적으로 모델이 로드될 것입니다.

.

# 3️⃣ LoRA 설정

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622


# 4️⃣ 한국어 데이터 준비

In [6]:
from datasets import Dataset

data = [
    {
        "instruction": "친절한 존댓말로 설명하세요",
        "input": "LLM이 뭐야?",
        "output": "LLM은 대규모 언어 모델로, 많은 텍스트 데이터를 학습하여 사람처럼 문장을 생성하는 AI입니다."
    },
    {
        "instruction": "초보자에게 쉽게 설명하세요",
        "input": "LoRA가 뭐야?",
        "output": "LoRA는 모델 전체를 학습하지 않고 일부만 학습해서 효율적으로 튜닝하는 방법입니다."
    }
]

dataset = Dataset.from_list(data)

# 5️⃣프롬프트 포맷 만들기 (핵심🔥)

In [7]:
def format_example(example):
    return f"""### 지시:
{example['instruction']}

### 입력:
{example['input']}

### 출력:
{example['output']}"""

dataset = dataset.map(lambda x: {"text": format_example(x)})

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

# 6️⃣ 토크나이징

In [8]:
# Llama 모델은 기본 pad_token이 없으므로 eos_token으로 설정해줍니다.
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # 모델이 손실(loss)을 계산할 수 있도록 input_ids를 labels로 복사해줍니다.
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

# 7️⃣ 학습 설정 (Hyperparameters)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results", # 결과물 저장경로
    per_device_train_batch_size=2, # 한 번에 학습할 데이터의 개수입니다. GPU 메모리(T4) 용량을 고려하여 2개씩 나누어 학습하도록 설정했습니다.

    # 학습 횟수를 대폭 늘려 모델이 패턴을 익히게 합니다. 데이터 양이 적을 때는 반복 횟수를 늘려야 모델이 패턴을 익힐 수 있습니다
    # 3회만 한경우 틀린답변을 하였으나 100회를 실행후엔 정확한 답변을 함
    num_train_epochs=100,

    logging_steps=10,
    save_strategy="no", # 이번 실습에서는 저장 공간을 아끼기 위해 중간 체크포인트를 저장하지 않도록 설정했습니다.
    learning_rate=2e-4, # 0.0002는 LoRA 학습에서 일반적으로 권장되는 값입니다.
    fp16=True # 16비트 부동소수점 연산을 사용하여 학습 속도를 높이고 메모리 사용량을 줄이는 기술입니다.
)

# 8️⃣ Trainer로 학습

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
10,18.475974
20,0.853133
30,0.279047
40,0.153132
50,0.072522
60,0.032909
70,0.018184
80,0.008219
90,0.004339
100,0.003469


TrainOutput(global_step=100, training_loss=1.9900927366316319, metrics={'train_runtime': 201.4087, 'train_samples_per_second': 0.993, 'train_steps_per_second': 0.497, 'total_flos': 4062128898048000.0, 'train_loss': 1.9900927366316319, 'epoch': 100.0})

# 9️⃣ 테스트 (추론)

In [23]:
def generate(instruction, input_text=""):
    # 학습할 때 사용한 프롬프트 포맷과 동일하게 맞춰줍니다.
    prompt = f"""### 지시:
{instruction}

### 입력:
{input_text}

### 출력:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # repetition_penalty를 추가하여 반복 문구를 방지합니다.
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 출력 부분만 잘라서 보여줍니다.
    return result.split("### 출력:")[-1].strip()



In [24]:
print(generate("친절한 존댓말로 설명하세요", "LLM이 뭐야?"))

LLM은 대규모 언어 모델로, 많은 텍스트 데이터를 학습해서 사람처럼 문장을 생성하는 AI입니다.


In [25]:
print(generate("초보자에게 쉽게 설명하세요", "LoRA가 뭐야?"))

LoRA는 모델 전체를 학습하지 않고 일부만 학습해서 효율적으로 튜닝하는 방법입니다.
